# 📝 Claim Decomposition Prompt Templates — Comparison Guide

This notebook compares four claim decomposition prompt templates for long-text UQ methods in `uqlm`.

**The four templates:**
- **`zhang_2025`** — Atomic Calibration ([Zhang et al., 2025](https://arxiv.org/abs/2410.13246)) — *current default*
- **`farquhar_2024`** — Semantic Entropy ([Farquhar et al., 2024, Nature](https://www.nature.com/articles/s41586-024-07421-0))
- **`mohri_2024`** — Conformal Factuality ([Mohri & Hashimoto, 2024, ICML](https://arxiv.org/abs/2402.10978))
- **`jiang_2024`** — Graph-based UQ ([Jiang et al., 2024](https://arxiv.org/abs/2410.20783))

## 1. Import and Setup

In [ ]:
import re
from langchain_openai import ChatOpenAI
from uqlm.utils.prompts import get_claim_breakdown_prompt, get_farquhar_2024_breakdown_prompt, get_mohri_2024_breakdown_prompt, get_jiang_2024_breakdown_prompt

# Set your API key
# os.environ["OPENAI_API_KEY"] = "your-key-here"

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

## 2. Template Descriptions

### zhang_2025 (default)
- **Granularity:** Very fine (most atomic)
- **Style:** Chain-of-thought with examples
- **Best for:** Maximum claim coverage, atomic fact scoring

### farquhar_2024
- **Granularity:** Coarse (fewest claims)
- **Style:** Zero-shot (NO examples) — exact Nature 2024 replication
- **Best for:** Semantic entropy, QA-based UQ
- **Note:** Retains pronouns, groups related facts

### mohri_2024
- **Granularity:** Fine (non-overlapping)
- **Style:** Few-shot with emphasis on independence
- **Best for:** Conformal prediction, best overall quality

### jiang_2024
- **Granularity:** Fine (no semantic repetition)
- **Style:** Few-shot optimized for graphs
- **Best for:** Graph-based UQ (LongTextGraph)

## 3. Prompt Details

### 3.1 zhang_2025 (Atomic Calibration)

**Instructions:**
- Breakdown passage into independent fact pieces
- Each fact piece should contain one single independent fact (subject + verb + object)
- Make each atomic fact self-contained (no pronouns)
- Output each fact starting with `###`

**Few-shot examples:** 3 examples provided (Michael Collins, League of Legends, Emory University)

**Example output format:**
```
### Michael Collins was born on October 31, 1930.
### Michael Collins is retired.
### Michael Collins is an American.
### Michael Collins was an astronaut.
```

---

### 3.2 farquhar_2024 (Semantic Entropy - Nature 2024)

**Instructions:**
- List specific factual propositions
- Be complete and don't leave any factual claims out
- Provide each claim as a separate sentence
- Output each fact starting with `###`

**Few-shot examples:** NONE (zero-shot)

**Key difference:** No examples → coarser granularity, may retain pronouns

---

### 3.3 mohri_2024 (Conformal Factuality)

**Instructions:**
- Breakdown into small, independent, non-overlapping claims
- Make sure each claim is small and non-overlapping
- Output each claim starting with `###`

**Few-shot examples:** 2 examples provided (Percy Bysshe Shelley, Eiffel Tower)

**Example output format:**
```
### Percy Bysshe Shelley was one of the major English Romantic poets.
### Percy Bysshe Shelley is regarded by critics as among the finest lyric poets in the English language.
### Percy Bysshe Shelley was a radical in his poetry.
```

---

### 3.4 jiang_2024 (Graph-based UQ)

**Instructions:**
- Deconstruct paragraph into smallest possible standalone self-contained facts
- No semantic repetition
- Output each fact starting with `###`

**Few-shot examples:** 2 examples provided (Albert Einstein, Great Wall of China)

**Example output format:**
```
### Albert Einstein was born in Germany.
### Albert Einstein was a theoretical physicist.
### Albert Einstein developed the theory of relativity.
### The theory of relativity is one of the two pillars of modern physics.
```

## 4. Example: Marie Curie Biography

In [11]:
EXAMPLE_TEXT = """
Marie Curie was a Polish-French physicist and chemist who conducted pioneering research on radioactivity. 
She was the first woman to win a Nobel Prize, and the only person to win Nobel Prizes in two different sciences — 
Physics in 1903 and Chemistry in 1911. Born Maria Sklodowska on November 7, 1867, in Warsaw, she moved to Paris 
in 1891 to pursue her studies. Together with her husband Pierre Curie, she discovered the elements polonium and 
radium. She died on July 4, 1934, from aplastic anemia, believed to have been caused by prolonged exposure to radiation.
""".strip()

print(EXAMPLE_TEXT)

Marie Curie was a Polish-French physicist and chemist who conducted pioneering research on radioactivity. 
She was the first woman to win a Nobel Prize, and the only person to win Nobel Prizes in two different sciences — 
Physics in 1903 and Chemistry in 1911. Born Maria Sklodowska on November 7, 1867, in Warsaw, she moved to Paris 
in 1891 to pursue her studies. Together with her husband Pierre Curie, she discovered the elements polonium and 
radium. She died on July 4, 1934, from aplastic anemia, believed to have been caused by prolonged exposure to radiation.


In [12]:
def parse_claims(output: str) -> list:
    """Extract claims prefixed with ###"""
    pattern = r"###\s*(.+?)(?=\n\s*###|\n\s*$|$)"
    matches = re.findall(pattern, output, re.MULTILINE | re.DOTALL)
    return [re.sub(r"\s+", " ", m.strip()) for m in matches if m.strip().upper() != "NONE"]


async def decompose(prompt_fn, text: str, label: str):
    """Run decomposition and display results"""
    result = await llm.ainvoke(prompt_fn(text))
    claims = parse_claims(result.content)

    print(f"\n{'=' * 70}")
    print(f"{label}: {len(claims)} claims")
    print(f"{'=' * 70}")
    for i, claim in enumerate(claims, 1):
        print(f"{i:2d}. {claim}")

    return claims

## 5. Run All Four Templates

In [13]:
# zhang_2025
zhang_claims = await decompose(get_claim_breakdown_prompt, EXAMPLE_TEXT, "zhang_2025")


zhang_2025: 17 claims
 1. Marie Curie was a Polish-French physicist.
 2. Marie Curie was a chemist.
 3. Marie Curie conducted pioneering research on radioactivity.
 4. Marie Curie was the first woman to win a Nobel Prize.
 5. Marie Curie was the only person to win Nobel Prizes in two different sciences.
 6. Marie Curie won a Nobel Prize in Physics in 1903.
 7. Marie Curie won a Nobel Prize in Chemistry in 1911.
 8. Marie Curie was born Maria Sklodowska.
 9. Maria Sklodowska was born on November 7, 1867.
10. Maria Sklodowska was born in Warsaw.
11. Maria Sklodowska moved to Paris in 1891.
12. Maria Sklodowska moved to Paris to pursue her studies.
13. Marie Curie discovered the element polonium.
14. Marie Curie discovered the element radium.
15. Marie Curie died on July 4, 1934.
16. Marie Curie died from aplastic anemia.
17. Aplastic anemia was believed to have been caused by prolonged exposure to radiation.


In [14]:
# farquhar_2024 (zero-shot)
farquhar_claims = await decompose(get_farquhar_2024_breakdown_prompt, EXAMPLE_TEXT, "farquhar_2024")


farquhar_2024: 13 claims
 1. Marie Curie was a Polish-French physicist and chemist.
 2. She conducted pioneering research on radioactivity.
 3. She was the first woman to win a Nobel Prize.
 4. She was the only person to win Nobel Prizes in two different sciences.
 5. She won the Nobel Prize in Physics in 1903.
 6. She won the Nobel Prize in Chemistry in 1911.
 7. She was born Maria Sklodowska on November 7, 1867.
 8. She was born in Warsaw.
 9. She moved to Paris in 1891 to pursue her studies.
10. Together with her husband Pierre Curie, she discovered the elements polonium and radium.
11. She died on July 4, 1934.
12. She died from aplastic anemia.
13. Her aplastic anemia was believed to have been caused by prolonged exposure to radiation.


In [15]:
# mohri_2024
mohri_claims = await decompose(get_mohri_2024_breakdown_prompt, EXAMPLE_TEXT, "mohri_2024")


mohri_2024: 15 claims
 1. Marie Curie was a Polish-French physicist.
 2. Marie Curie was a chemist.
 3. Marie Curie conducted pioneering research on radioactivity.
 4. Marie Curie was the first woman to win a Nobel Prize.
 5. Marie Curie is the only person to win Nobel Prizes in two different sciences.
 6. Marie Curie won the Nobel Prize in Physics in 1903.
 7. Marie Curie won the Nobel Prize in Chemistry in 1911.
 8. Marie Curie was born Maria Sklodowska on November 7, 1867.
 9. Marie Curie was born in Warsaw.
10. Marie Curie moved to Paris in 1891 to pursue her studies.
11. Marie Curie discovered the element polonium.
12. Marie Curie discovered the element radium.
13. Marie Curie died on July 4, 1934.
14. Marie Curie died from aplastic anemia.
15. Marie Curie's aplastic anemia was believed to have been caused by prolonged exposure to radiation.


In [16]:
# jiang_2024
jiang_claims = await decompose(get_jiang_2024_breakdown_prompt, EXAMPLE_TEXT, "jiang_2024")


jiang_2024: 17 claims
 1. Marie Curie was a physicist and chemist.
 2. Marie Curie was Polish-French.
 3. Marie Curie conducted pioneering research on radioactivity.
 4. Marie Curie was the first woman to win a Nobel Prize.
 5. Marie Curie was the only person to win Nobel Prizes in two different sciences.
 6. Marie Curie won the Nobel Prize in Physics in 1903.
 7. Marie Curie won the Nobel Prize in Chemistry in 1911.
 8. Marie Curie was born Maria Sklodowska.
 9. Marie Curie was born on November 7, 1867.
10. Marie Curie was born in Warsaw.
11. Marie Curie moved to Paris in 1891.
12. Marie Curie moved to Paris to pursue her studies.
13. Marie Curie discovered the elements polonium and radium.
14. Marie Curie worked with her husband Pierre Curie.
15. Marie Curie died on July 4, 1934.
16. Marie Curie died from aplastic anemia.
17. Aplastic anemia was believed to be caused by prolonged exposure to radiation.


## 6. Summary Comparison

In [17]:
print("\nClaim Count Summary:")
print("-" * 40)
print(f"zhang_2025    : {len(zhang_claims)} claims")
print(f"farquhar_2024 : {len(farquhar_claims)} claims (zero-shot)")
print(f"mohri_2024    : {len(mohri_claims)} claims")
print(f"jiang_2024    : {len(jiang_claims)} claims")


Claim Count Summary:
----------------------------------------
zhang_2025    : 17 claims
farquhar_2024 : 13 claims (zero-shot)
mohri_2024    : 15 claims
jiang_2024    : 17 claims


## 7. Expected Results (GPT-4o-mini, temp=0)

| Template | # Claims | Characteristics |
|----------|----------|----------------|
| `zhang_2025` | 17 | Most atomic; each date/role/discovery separate |
| `farquhar_2024` | 10 | Coarsest; retains pronouns ("She", "Her") |
| `mohri_2024` | 15 | Fine-grained; best balance |
| `jiang_2024` | 17 | Fine-grained; no redundancy |

## 8. Usage in uqlm

```python
from uqlm import LongTextUQ

# Use string key
scorer = LongTextUQ(
    llm=llm,
    claim_decomposition_prompt="mohri_2024"
)

# Or use custom callable
def my_prompt(response: str) -> str:
    return f"Break into facts:\n{response}\nFormat: ### <claim>"

scorer = LongTextUQ(
    llm=llm,
    claim_decomposition_prompt=my_prompt
)
```

## 9. References

- Zhang et al. (2025). *Atomic Calibration of LLMs in Long-Form Generations.* arXiv:2410.13246
- Farquhar et al. (2024). *Detecting hallucinations in large language models using semantic entropy.* Nature, 630:625-630
- Mohri & Hashimoto (2024). *Language Models with Conformal Factuality Guarantees.* ICML 2024
- Jiang et al. (2024). *Graph-based Uncertainty Metrics for Long-form Language Model Outputs.* arXiv:2410.20783